# confusion_matrix — 학습된 best.pt로 val셋 혼동행렬 생성

`colab_run.ipynb`는 건드리지 않고, **이미 학습해 둔 `best.pt`만 불러와** val셋으로 한 번 더 추론해서
혼동행렬(confusion matrix)을 새로 그립니다.

- 결과 그림은 `colab_run.ipynb`와 **똑같이** `outputs/yolo/<RUN_NAME>/` (예: `test_run`) 안에 저장됩니다.
- 어떤 클래스를 어떤 클래스로 헷갈렸는지(오분류) 표로도 출력합니다.

**사전 준비:** 런타임 → 런타임 유형 변경 → **GPU**

**전제:** `colab_run.ipynb`로 학습이 끝나 Drive `outputs/yolo/<RUN_NAME>/weights/best.pt`가 존재해야 합니다.

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 설정 — **이 셀만 수정하세요**

- `DRIVE_PROJECT_DIR`: 구글 드라이브의 프로젝트 루트 (`colab_run.ipynb`와 동일하게)
- `RUN_NAME`: 혼동행렬을 보고 싶은 **학습 실행 이름**. 그 `best.pt`가 `outputs/yolo/<RUN_NAME>/weights/best.pt`에 있어야 함
- `SPLIT`: 혼동행렬을 계산할 데이터 분할 (`val` 권장. 라벨이 있는 `train`도 가능)

In [ ]:
import os

# ─── 여기만 수정 ─────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/베이스라인_코랩버전v1.4'

RUN_NAME = 'test_run'   # 혼동행렬을 볼 학습 실행 이름 (best.pt가 있는 곳)
SPLIT    = 'val'        # 'val' 권장 (라벨 있는 분할)
# ────────────────────────────────────────────

DRIVE_BASELINE = os.path.join(DRIVE_PROJECT_DIR, 'baseline')
DRIVE_OUTPUTS  = os.path.join(DRIVE_PROJECT_DIR, 'outputs')
DRIVE_DATA_ZIP = os.path.join(DRIVE_PROJECT_DIR, 'sprint_ai_project1_data.zip')
DRIVE_RUN_DIR  = os.path.join(DRIVE_OUTPUTS, 'yolo', RUN_NAME)

LOCAL_PROJECT_DIR = '/content/project'
LOCAL_BASELINE    = os.path.join(LOCAL_PROJECT_DIR, 'baseline')
LOCAL_OUTPUTS     = os.path.join(LOCAL_BASELINE, 'outputs')
LOCAL_DATA_DIR    = os.path.join(LOCAL_PROJECT_DIR, 'sprint_ai_project1_data')

TRAIN_OUTPUT_DIR = f'outputs/yolo/{RUN_NAME}'
BEST_WEIGHTS     = f'{TRAIN_OUTPUT_DIR}/weights/best.pt'

print('--- Drive ---')
for p in [DRIVE_PROJECT_DIR, DRIVE_BASELINE, DRIVE_DATA_ZIP, DRIVE_RUN_DIR]:
    print(f'  exists={os.path.exists(p)}  {p}')
print('--- Run config ---')
print(f'  RUN_NAME = {RUN_NAME}   SPLIT = {SPLIT}')
print(f'  best.pt(Drive) exists = {os.path.exists(os.path.join(DRIVE_RUN_DIR, "weights", "best.pt"))}')

## 3. Drive → Colab 로컬 복제 (baseline + 데이터)

`colab_run.ipynb`와 동일. 이미 있으면 skip (재실행 안전).

In [ ]:
import os, shutil, subprocess, time

os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)

def copy_dir(src, dst, label):
    if not os.path.exists(src):
        print(f'[skip] {label}: Drive에 없음'); return
    if os.path.exists(dst):
        print(f'[skip] {label}: 이미 로컬에 있음'); return
    t0 = time.time()
    shutil.copytree(src, dst)
    print(f'[ok]   {label} 복사 완료 ({time.time()-t0:.1f}s)')

copy_dir(DRIVE_BASELINE, LOCAL_BASELINE, 'baseline/')

if os.path.exists(LOCAL_DATA_DIR):
    print('[skip] dataset: 이미 해제됨')
else:
    assert os.path.exists(DRIVE_DATA_ZIP), f'데이터 zip 없음: {DRIVE_DATA_ZIP}'
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    print('[..] dataset 해제 중 (수 분 소요)...')
    t0 = time.time()
    subprocess.run(['unzip', '-q', DRIVE_DATA_ZIP, '-d', LOCAL_DATA_DIR], check=True)
    print(f'[ok] dataset 해제 완료 ({time.time()-t0:.1f}s)')

## 4. 작업 디렉토리 + config 경로 갱신

In [ ]:
import os, yaml

os.chdir(LOCAL_BASELINE)
print('CWD:', os.getcwd())

cfg_path = 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']     = LOCAL_DATA_DIR
cfg['data']['processed_dir'] = './data/processed'
cfg['data']['num_workers']   = 2

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])

## 5. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics

## 6. 데이터 준비 (전처리 + YOLO 변환, 최초 1회)

val셋으로 추론하려면 `data/yolo/dataset.yaml`이 있어야 합니다. 이미 만들어져 있으면 빠르게 넘어갑니다.

In [ ]:
import os

if os.path.exists('data/yolo/dataset.yaml'):
    print('[skip] data/yolo/dataset.yaml 이미 존재')
else:
    !python scripts/preprocess.py
    !python scripts/convert_to_yolo.py

## 7. best.pt 복원 (Drive → 로컬)

Drive `outputs/yolo/<RUN_NAME>/weights/best.pt`를 로컬 같은 경로로 내려받습니다.

In [ ]:
import os, shutil

src_best = os.path.join(DRIVE_RUN_DIR, 'weights', 'best.pt')
assert os.path.exists(src_best), f'Drive에 best.pt 없음: {src_best}'

os.makedirs(os.path.dirname(BEST_WEIGHTS), exist_ok=True)
shutil.copy2(src_best, BEST_WEIGHTS)
print(f'[ok] best.pt 복원 → {BEST_WEIGHTS}  (exists={os.path.exists(BEST_WEIGHTS)})')

## 8. val셋 추론 → 혼동행렬 생성 + 오분류 표

`project`/`name`을 `outputs/yolo/<RUN_NAME>`으로 지정해, 혼동행렬 그림이 **학습 결과와 같은 폴더**에 저장(갱신)됩니다.
동시에 대각선(정답)을 지운 **오분류만 추린 표**를 출력합니다.

In [ ]:
from ultralytics import YOLO
import pandas as pd
import numpy as np

model = YOLO(BEST_WEIGHTS)

# project/name 지정 → outputs/yolo/<RUN_NAME>/ 에 confusion_matrix*.png 저장(갱신)
results = model.val(
    data='data/yolo/dataset.yaml',
    split=SPLIT,
    project='outputs/yolo',
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    verbose=False,
)

cm = results.confusion_matrix.matrix
names = list(model.names.values()) + ['background']

df = pd.DataFrame(cm.astype(int), index=names, columns=names)

# 대각선(정답) 제거 → 틀린 케이스만 남김
np.fill_diagonal(df.values, 0)
wrong = df[df.sum(axis=1) > 0]
wrong = wrong[wrong.columns[(wrong > 0).any()]]

print('=== 오분류 행렬 (행=실제, 열=예측) ===')
print(wrong)
print(f'\n저장 위치: outputs/yolo/{RUN_NAME}/confusion_matrix_normalized.png')

## 9. 정규화 혼동행렬 이미지 표시

In [ ]:
from IPython.display import Image
Image(f'outputs/yolo/{RUN_NAME}/confusion_matrix_normalized.png')

## 10. (선택) 갱신된 혼동행렬을 Drive로 백업

방금 새로 그린 PNG를 Drive `outputs/yolo/<RUN_NAME>/`에도 남기고 싶을 때 실행하세요.

In [ ]:
import shutil, os, time

src = os.path.join(LOCAL_OUTPUTS, 'yolo', RUN_NAME)
dst = DRIVE_RUN_DIR
t0 = time.time()
for fn in ['confusion_matrix.png', 'confusion_matrix_normalized.png']:
    s = os.path.join(src, fn)
    if os.path.exists(s):
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(s, os.path.join(dst, fn))
        print(f'[ok] {fn} → Drive')
print(f'백업 완료 ({time.time()-t0:.1f}s): {dst}')